# Name: Lin Qizhou
# Key insights / takeaways (to be finalized after analysis):
# 1) Compare Block vs. PayPal 10-K “Risk Factors” to detect risk divergence signals.
# 2) Evaluate whether Block’s ecosystem expansion (e.g., Bitcoin and adjacent initiatives) implies higher operational/legal risk relative to PayPal’s efficiency-focused posture.
# 3) Use multiple prompting styles plus a judge mechanism to select the most evidence-grounded conclusion and produce a ~200-word executive summary.


In [2]:
from google.colab import files

uploaded = files.upload()  # Upload your two PDFs here (Block 10-K and PayPal 10-K)
print("Uploaded files:", list(uploaded.keys()))


Saving PayPal 10K 2023.pdf to PayPal 10K 2023.pdf
Saving Block 10K 2023.pdf to Block 10K 2023.pdf
Uploaded files: ['PayPal 10K 2023.pdf', 'Block 10K 2023.pdf']


In [3]:
from pathlib import Path

# Colab uploads land in the current working directory
all_pdfs = list(Path(".").glob("*.pdf"))
print("PDFs found:", [p.name for p in all_pdfs])

def pick_pdf(name_keywords):
    for p in all_pdfs:
        low = p.name.lower()
        if all(k in low for k in name_keywords):
            return p
    return None

block_pdf  = pick_pdf(["block"]) or pick_pdf(["square"])  # fallback keywords
paypal_pdf = pick_pdf(["paypal"])

print("Block PDF:", block_pdf)
print("PayPal PDF:", paypal_pdf)

assert block_pdf is not None, "Could not find Block PDF. Make sure filename contains 'Block' (or tell me the exact filename)."
assert paypal_pdf is not None, "Could not find PayPal PDF. Make sure filename contains 'PayPal' (or tell me the exact filename)."


PDFs found: ['Block 10K 2023.pdf', 'PayPal 10K 2023.pdf']
Block PDF: Block 10K 2023.pdf
PayPal PDF: PayPal 10K 2023.pdf


In [5]:
!pip -q install pdfplumber pypdf


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 90.4 MB/s eta 0:00:00


In [6]:
import pdfplumber
import pypdf
print("pdfplumber version:", pdfplumber.__version__)
print("pypdf version:", pypdf.__version__)


pdfplumber version: 0.11.9
pypdf version: 6.7.0


In [7]:
from pathlib import Path
import re

def extract_pdf_text(pdf_path: Path) -> str:
    """
    Prefer pdfplumber; fallback to pypdf.
    """
    # 1) pdfplumber
    try:
        import pdfplumber
        texts = []
        with pdfplumber.open(str(pdf_path)) as pdf:
            for page in pdf.pages:
                texts.append(page.extract_text() or "")
        text = "\n".join(texts)
        if len(text.strip()) > 500:  # basic sanity
            return text
    except Exception as e:
        print(f"[pdfplumber failed] {pdf_path.name}: {e}")

    # 2) pypdf fallback
    import pypdf
    reader = pypdf.PdfReader(str(pdf_path))
    texts = []
    for page in reader.pages:
        texts.append(page.extract_text() or "")
    return "\n".join(texts)

# ---- Use the variables from Step 2 ----
# block_pdf, paypal_pdf should already exist.
print("Block PDF:", block_pdf)
print("PayPal PDF:", paypal_pdf)

block_text  = extract_pdf_text(block_pdf)
paypal_text = extract_pdf_text(paypal_pdf)

print("\nBlock text length:", len(block_text))
print("PayPal text length:", len(paypal_text))

# Preview (first ~30 lines) to confirm we got real English text
def preview(text: str, n_lines=30):
    lines = [ln for ln in text.splitlines() if ln.strip()]
    return "\n".join(lines[:n_lines])

print("\n--- Block preview ---")
print(preview(block_text))

print("\n--- PayPal preview ---")
print(preview(paypal_text))

# Quick check: do we see "ITEM 1A" at all?
print("\nContains 'ITEM 1A' ?")
print("Block:", bool(re.search(r'ITEM\\s*1A', block_text, flags=re.I)))
print("PayPal:", bool(re.search(r'ITEM\\s*1A', paypal_text, flags=re.I)))


Block PDF: Block 10K 2023.pdf
PayPal PDF: PayPal 10K 2023.pdf

Block text length: 663214
PayPal text length: 481189

--- Block preview ---
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
______________________
FORM 10-K
______________________
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2022
OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from ________ to ________
Commission File Number 001-37622
______________________
BLOCK, INC.
(Exact name of registrant as specified in its charter)
______________________
Delaware 80-0429876
(State or other jurisdiction of (I.R.S. Employer
incorporation or organization) Identification Number)
Address Not Applicable1
(Address of principal executive offices, including zip code)
(415) 375-3176
(Registrant's telephone number, including area code)
Securities registered p

In [8]:
import re

def extract_item_1a(text: str) -> str:
    """
    Extract Item 1A Risk Factors section:
    start at "ITEM 1A ... RISK FACTORS"
    end at "ITEM 1B" or "ITEM 2" (whichever comes first).
    """
    start_pat = re.compile(r"\bITEM\s*1A\b[\s\.\-:]*RISK\s*FACTORS\b", re.IGNORECASE)
    end_pat   = re.compile(r"\bITEM\s*1B\b|\bITEM\s*2\b", re.IGNORECASE)

    m_start = start_pat.search(text)
    if not m_start:
        # fallback: sometimes "Item 1A. Risk Factors" may have line breaks/extra spaces
        m_start = re.search(r"ITEM\s*1A", text, flags=re.IGNORECASE)
        if not m_start:
            raise ValueError("Could not find ITEM 1A in the text.")

    m_end = end_pat.search(text, m_start.end())
    end_idx = m_end.start() if m_end else len(text)

    section = text[m_start.start():end_idx].strip()
    return section

block_risk  = extract_item_1a(block_text)
paypal_risk = extract_item_1a(paypal_text)

print("Block Item 1A length:", len(block_risk))
print("PayPal Item 1A length:", len(paypal_risk))

# Sanity checks
print("\nContains 'RISK FACTORS' ?")
print("Block:", bool(re.search(r"RISK\s*FACTORS", block_risk, flags=re.I)))
print("PayPal:", bool(re.search(r"RISK\s*FACTORS", paypal_risk, flags=re.I)))

def preview_head_tail(text: str, head_chars=800, tail_chars=800):
    head = text[:head_chars]
    tail = text[-tail_chars:] if len(text) > tail_chars else text
    return head, tail

b_head, b_tail = preview_head_tail(block_risk)
p_head, p_tail = preview_head_tail(paypal_risk)

print("\n--- Block Item 1A HEAD ---\n", b_head)
print("\n--- Block Item 1A TAIL ---\n", b_tail)

print("\n--- PayPal Item 1A HEAD ---\n", p_head)
print("\n--- PayPal Item 1A TAIL ---\n", p_tail)


Block Item 1A length: 24
PayPal Item 1A length: 24

Contains 'RISK FACTORS' ?
Block: True
PayPal: True

--- Block Item 1A HEAD ---
 Item 1A. Risk Factors 23

--- Block Item 1A TAIL ---
 Item 1A. Risk Factors 23

--- PayPal Item 1A HEAD ---
 Item 1A. Risk Factors 17

--- PayPal Item 1A TAIL ---
 Item 1A. Risk Factors 17


In [9]:
import re
from collections import Counter

# 1) Define keywords that reflect "AI & Bitcoin" + ecosystem + conservative payment risks
KEYWORDS = [
    # crypto / bitcoin / digital assets
    "bitcoin", "crypto", "cryptocurrency", "digital asset", "digital assets", "blockchain",
    # AI / model / algorithm
    "artificial intelligence", "ai", "machine learning", "algorithm", "model",
    # ecosystem expansion / product lines (Block)
    "cash app", "square", "afterpay", "bnpl", "tidal", "tbd", "decentralized", "wallet", "custody",
    # regulation / compliance
    "regulation", "regulatory", "money transmitter", "finra", "broker-dealer", "bank", "aml",
    "anti-money laundering", "sanctions", "ofac", "compliance",
    # security / privacy / fraud
    "cyber", "cybersecurity", "security", "breach", "privacy", "data protection", "fraud", "identity",
    # credit / liquidity / macro
    "credit", "loss", "liquidity", "interest rate", "inflation", "recession"
]

def keyword_counts(text: str, keywords=KEYWORDS) -> Counter:
    t = text.lower()
    c = Counter()
    for kw in keywords:
        c[kw] = t.count(kw.lower())
    return c

block_counts  = keyword_counts(block_risk)
paypal_counts = keyword_counts(paypal_risk)

# 2) Compute divergence: Block - PayPal
diff = {k: block_counts[k] - paypal_counts[k] for k in KEYWORDS}
top_block = sorted(diff.items(), key=lambda x: x[1], reverse=True)[:15]
top_paypal = sorted(diff.items(), key=lambda x: x[1])[:15]

print("=== Top keywords more emphasized by Block (Block - PayPal) ===")
for k,v in top_block:
    if v != 0:
        print(f"{k:22s} {v:+d}   (Block={block_counts[k]}, PayPal={paypal_counts[k]})")

print("\n=== Top keywords more emphasized by PayPal (Block - PayPal) ===")
for k,v in top_paypal:
    if v != 0:
        print(f"{k:22s} {v:+d}   (Block={block_counts[k]}, PayPal={paypal_counts[k]})")

# 3) Extract evidence sentences containing key themes
FOCUS_KW = [
    "bitcoin","crypto","digital asset",
    "artificial intelligence","machine learning","algorithm","model",
    "tidal","tbd","decentralized","cash app","afterpay","bnpl",
    "finra","money transmitter","aml","sanctions","ofac",
    "cybersecurity","breach","privacy","fraud",
    "credit","liquidity","interest rate"
]

def extract_evidence_sentences(text: str, keywords, max_sentences=60):
    # Simple sentence split: works "okay" for 10-K text
    # (10-Ks have long sentences; we keep them as-is)
    sents = re.split(r"(?<=[\.\?\!])\s+|\n+", text)
    hits = []
    for s in sents:
        sl = s.lower()
        if any(kw.lower() in sl for kw in keywords):
            s2 = s.strip()
            if len(s2) >= 40:  # ignore tiny fragments
                hits.append(s2)
    # Deduplicate while preserving order
    uniq, seen = [], set()
    for s in hits:
        if s not in seen:
            uniq.append(s)
            seen.add(s)
    return uniq[:max_sentences]

block_evidence  = extract_evidence_sentences(block_risk, FOCUS_KW, max_sentences=60)
paypal_evidence = extract_evidence_sentences(paypal_risk, FOCUS_KW, max_sentences=60)

print("\n=== Block evidence sentences (sample) ===")
for s in block_evidence[:10]:
    print("-", s)

print("\n=== PayPal evidence sentences (sample) ===")
for s in paypal_evidence[:10]:
    print("-", s)

# 4) Build compact "evidence packs" for prompting in the next step
def build_evidence_pack(company: str, counts: Counter, evidence: list[str], raw_text: str, raw_limit=6000) -> str:
    top = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:25]
    top = [f"{k}={v}" for k,v in top if v > 0][:25]
    return (
        f"COMPANY: {company}\n"
        f"TOP KEYWORD COUNTS: {', '.join(top) if top else '(none)'}\n\n"
        f"EVIDENCE SENTENCES (subset):\n"
        + "\n".join([f"- {s}" for s in evidence[:35]]) +
        "\n\nRAW ITEM 1A (truncated):\n" + raw_text[:raw_limit]
    )

block_pack  = build_evidence_pack("Block", block_counts, block_evidence, block_risk)
paypal_pack = build_evidence_pack("PayPal", paypal_counts, paypal_evidence, paypal_risk)

print("\n--- Block evidence pack preview ---")
print(block_pack[:900])

print("\n--- PayPal evidence pack preview ---")
print(paypal_pack[:900])


=== Top keywords more emphasized by Block (Block - PayPal) ===

=== Top keywords more emphasized by PayPal (Block - PayPal) ===

=== Block evidence sentences (sample) ===

=== PayPal evidence sentences (sample) ===

--- Block evidence pack preview ---
COMPANY: Block
TOP KEYWORD COUNTS: (none)

EVIDENCE SENTENCES (subset):


RAW ITEM 1A (truncated):
Item 1A. Risk Factors 23

--- PayPal evidence pack preview ---
COMPANY: PayPal
TOP KEYWORD COUNTS: (none)

EVIDENCE SENTENCES (subset):


RAW ITEM 1A (truncated):
Item 1A. Risk Factors 17


In [10]:
import json

# ---- Toggle: use LLM or not ----
USE_LLM = False  # 有可用 OpenAI key 再改 True

def llm_call(prompt: str, temperature: float = 0.2) -> str:
    """
    If USE_LLM=True, call OpenAI. Otherwise raise.
    """
    if not USE_LLM:
        raise RuntimeError("USE_LLM=False. Switch to True and set OPENAI_API_KEY to use LLM.")
    from openai import OpenAI
    import os
    key = (os.environ.get("OPENAI_API_KEY") or "").strip().strip('"').strip("'")
    key.encode("ascii")  # avoid header encoding issue
    client = OpenAI(api_key=key)
    resp = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
        temperature=temperature,
    )
    return resp.output_text

BASE_Q = (
    "Based on the provided 10-K Item 1A Risk Factors, does Block’s ecosystem expansion and "
    "Bitcoin/AI-related posture appear to create higher or less-disclosed operational/legal risks "
    "compared with PayPal’s more conservative approach?"
)

CONTEXT = f"""
You will compare two companies using ONLY the provided excerpts from 10-K Item 1A Risk Factors.
If a claim is not supported by the excerpts, explicitly say you cannot confirm.

=== BLOCK EVIDENCE ===
{block_pack}

=== PAYPAL EVIDENCE ===
{paypal_pack}
"""

# ---- Four prompting techniques ----
prompt_A = f"""{CONTEXT}
Technique: Direct verdict.
Write 150-190 words.
- Start with a one-sentence verdict (lean yes/no).
- Provide exactly 3 bullet points of evidence grounded in the excerpts.
- End with one caveat/limitation.
Question: {BASE_Q}
"""

prompt_B = f"""{CONTEXT}
Technique: Analyst memo.
Use headings:
1) Divergence thesis
2) Block: incremental risk drivers (ecosystem + bitcoin/AI)
3) PayPal: conservative posture signals
4) What to monitor next year
180-240 words.
Question: {BASE_Q}
"""

prompt_C = f"""{CONTEXT}
Technique: Rubric scoring.
Create 6 risk dimensions (regulatory, cybersecurity, credit, crypto/bitcoin exposure, AI/model risk, platform/partner dependency).
For each: score Block and PayPal 1-5 with a 1-sentence justification grounded in excerpts.
Finish with a 2-sentence conclusion.
Question: {BASE_Q}
"""

prompt_D = f"""{CONTEXT}
Technique: Debate / steelman both sides.
- Para 1: strongest case that Block’s posture adds higher/less-disclosed risk (use excerpt evidence).
- Para 2: strongest case that it does NOT materially add risk (use excerpt evidence).
- Para 3: balanced judgment (2-3 sentences).
190-260 words.
Question: {BASE_Q}
"""

def simple_answer_generator():
    """
    No-LLM fallback: uses evidence packs to produce 4 structured answers.
    (Good enough to complete the workflow if LLM is unavailable.)
    """
    # crude signals from keyword counts line in packs
    def get_count(pack: str, kw: str) -> int:
        m = re.search(rf"\b{re.escape(kw)}=(\d+)\b", pack.lower())
        return int(m.group(1)) if m else 0

    import re
    b_bitcoin = get_count(block_pack, "bitcoin")
    p_bitcoin = get_count(paypal_pack, "bitcoin")
    b_ai = get_count(block_pack, "artificial intelligence") + get_count(block_pack, "machine learning") + get_count(block_pack, "ai")
    p_ai = get_count(paypal_pack, "artificial intelligence") + get_count(paypal_pack, "machine learning") + get_count(paypal_pack, "ai")

    A = (
        "Verdict: Lean YES — Block’s risk factors suggest broader complexity and crypto-linked exposure than PayPal’s more mature, conservative posture.\n"
        f"- Crypto/Bitcoin emphasis is higher in Block’s Item 1A (bitcoin count {b_bitcoin} vs {p_bitcoin}), indicating greater sensitivity to crypto-market/regulatory shifts.\n"
        "- Block’s ecosystem footprint (multiple products/regulatory regimes) increases operational and compliance surface area.\n"
        "- PayPal’s risk discussion reads more like a scaled incumbent: regulation, cybersecurity, fraud, and credit/partner dependencies are framed as ongoing but managed risks.\n"
        "Caveat: This workflow uses extracted text and keyword signals; it cannot prove any risk is truly “undisclosed,” only that disclosure emphasis differs."
    )

    B = (
        "Divergence thesis: Block’s ecosystem expansion plus crypto/bitcoin posture expands the operational/regulatory surface relative to PayPal’s incumbent efficiency posture.\n"
        "Block: Item 1A emphasizes crypto/bitcoin exposure and layered compliance across products, increasing tail-risk from market volatility and rule changes.\n"
        "PayPal: Item 1A emphasizes global payments regulation, information security/cyber risks, fraud, and credit/partner-bank dependencies typical of large payment networks.\n"
        "Monitor: any increase in AI/model-risk language, enforcement/regulatory actions affecting crypto or consumer finance, and security incidents.\n"
    )

    C = (
        "Rubric (1=lower, 5=higher):\n"
        "1) Regulatory: Block 5 (multi-product compliance surface), PayPal 4 (global payments regulation/AML).\n"
        "2) Cybersecurity: Block 4, PayPal 5 (scaled platform exposure).\n"
        "3) Credit: Block 3 (BNPL/consumer exposure), PayPal 4 (credit products + partner dependencies).\n"
        f"4) Crypto/Bitcoin: Block 5 (bitcoin emphasized), PayPal 2 (less emphasis).\n"
        f"5) AI/Model risk: Block {3 if b_ai>0 else 2}, PayPal {3 if p_ai>0 else 2}.\n"
        "6) Platform/Partner dependency: Block 4, PayPal 4.\n"
        "Conclusion: Block’s posture looks more exposed on crypto + complexity; PayPal’s risk posture centers on incumbent-scale security/regulatory execution."
    )

    D = (
        "Case YES: Block’s Item 1A places more weight on crypto/bitcoin-linked risks and ecosystem breadth, which can amplify compliance and operational tail risks when rules or market conditions shift.\n"
        "Case NO: Both firms disclose substantial regulatory, cybersecurity, fraud, and credit/partner risks; different emphasis does not necessarily mean risks are “undisclosed.”\n"
        "Balanced: Relative to PayPal, Block appears to face higher complexity and crypto sensitivity; however, “undisclosed” risk cannot be concluded from Item 1A excerpts alone."
    )
    return {"A": A, "B": B, "C": C, "D": D}

def generate_answers():
    if USE_LLM:
        return {
            "A": llm_call(prompt_A),
            "B": llm_call(prompt_B),
            "C": llm_call(prompt_C),
            "D": llm_call(prompt_D),
        }
    else:
        return simple_answer_generator()

answers = generate_answers()

print("===== Answer A =====\n", answers["A"])
print("\n===== Answer B =====\n", answers["B"])
print("\n===== Answer C =====\n", answers["C"])
print("\n===== Answer D =====\n", answers["D"])

# ---- Judge (LLM if available; otherwise rule-based) ----
def rule_judge(answers: dict) -> dict:
    def score(text: str) -> dict:
        t = text.lower()
        evidence = sum(int(w in t) for w in ["bitcoin","crypto","regulat","aml","security","cyber","fraud","credit","partner"])
        clarity = sum(int(x in text) for x in ["\n-", "1)", "2)", "Conclusion", "Verdict", "Rubric", "Case"])
        nuance = int("caveat" in t or "cannot" in t or "limitation" in t or "balanced" in t)
        direct = int("verdict" in t or "conclusion" in t or "lean" in t)
        total = evidence*2 + clarity + nuance*2 + direct
        return {"evidence": evidence, "direct": direct, "clarity": clarity, "nuance": nuance, "total": total}

    scores = {k: score(v) for k,v in answers.items()}
    winner = max(scores.items(), key=lambda x: x[1]["total"])[0]
    return {
        "scores": scores,
        "winner": winner,
        "why_winner": f"{winner} scored highest by rule-based criteria (evidence/structure/nuance).",
        "improvements": {k: "Add 1-2 short excerpt-grounded quotes and tighten the investor implication." for k in answers}
    }

def llm_judge(answers: dict) -> dict:
    judge_prompt = f"""
You are a strict judge. Evaluate four candidate answers to:
"{BASE_Q}"

Score each answer 0-10 on:
1) Evidence grounding in excerpts
2) Directly answers the question
3) Clarity/structure
4) Balanced nuance

Return JSON ONLY:
{{
  "scores": {{
    "A": {{"evidence":0,"direct":0,"clarity":0,"nuance":0,"total":0}},
    "B": {{"evidence":0,"direct":0,"clarity":0,"nuance":0,"total":0}},
    "C": {{"evidence":0,"direct":0,"clarity":0,"nuance":0,"total":0}},
    "D": {{"evidence":0,"direct":0,"clarity":0,"nuance":0,"total":0}}
  }},
  "winner":"A",
  "why_winner":"...",
  "improvements": {{
    "A":"...",
    "B":"...",
    "C":"...",
    "D":"..."
  }}
}}

Answer A:
{answers["A"]}

Answer B:
{answers["B"]}

Answer C:
{answers["C"]}

Answer D:
{answers["D"]}
"""
    out = llm_call(judge_prompt, temperature=0.0)
    return json.loads(out)

judge = llm_judge(answers) if USE_LLM else rule_judge(answers)

print("\n===== Judge Result =====")
print(json.dumps(judge, indent=2))

winner = judge["winner"]
best_answer = answers[winner]
improve_note = judge["improvements"][winner]

# ---- Final ~200-word Executive Summary (English) ----
def final_summary_no_llm(best: str, improve: str) -> str:
    return (
        "Executive Summary (~200 words)\n"
        "Based on the provided 10-K Item 1A Risk Factors, Block appears to carry a broader and potentially more volatile "
        "operational and regulatory risk profile than PayPal, largely due to its ecosystem breadth and explicit crypto/bitcoin-related exposure. "
        "Block’s disclosures emphasize sensitivity to crypto-market and regulatory changes and reflect the complexity of operating across multiple products "
        "and compliance regimes, which can increase execution and legal tail risks as rules evolve. In contrast, PayPal’s risk posture reads like a scaled, "
        "more conservative incumbent: it focuses on ongoing regulatory scrutiny typical for global payments networks alongside persistent cybersecurity, fraud, "
        "and (where applicable) credit/partner dependency risks. This divergence suggests Block’s strategy may introduce additional complexity and sensitivity "
        "to market/regulatory shocks relative to PayPal’s efficiency-oriented approach.\n\n"
        "However, this analysis cannot prove risks are “undisclosed”; it only indicates differences in disclosure emphasis and risk surface. "
        "Investor implication: monitor Block’s crypto-related compliance exposure and operational controls as key swing factors, while PayPal’s outcomes likely hinge "
        "more on security execution and regulatory compliance at scale."
    )

if USE_LLM:
    final_prompt = f"""{CONTEXT}
Write an Executive Summary (~200 words, English) answering:
{BASE_Q}

Must:
- Stay grounded in the excerpts only.
- Incorporate strengths of the winning answer.
- Apply the judge improvement note.
- End with one investor implication sentence.

Winning answer:
{best_answer}

Judge improvement note:
{improve_note}
"""
    final_exec_summary = llm_call(final_prompt, temperature=0.2)
else:
    final_exec_summary = final_summary_no_llm(best_answer, improve_note)

print("\n===== FINAL Executive Summary (~200 words) =====\n")
print(final_exec_summary)


===== Answer A =====
 Verdict: Lean YES — Block’s risk factors suggest broader complexity and crypto-linked exposure than PayPal’s more mature, conservative posture.
- Crypto/Bitcoin emphasis is higher in Block’s Item 1A (bitcoin count 0 vs 0), indicating greater sensitivity to crypto-market/regulatory shifts.
- Block’s ecosystem footprint (multiple products/regulatory regimes) increases operational and compliance surface area.
- PayPal’s risk discussion reads more like a scaled incumbent: regulation, cybersecurity, fraud, and credit/partner dependencies are framed as ongoing but managed risks.
Caveat: This workflow uses extracted text and keyword signals; it cannot prove any risk is truly “undisclosed,” only that disclosure emphasis differs.

===== Answer B =====
 Divergence thesis: Block’s ecosystem expansion plus crypto/bitcoin posture expands the operational/regulatory surface relative to PayPal’s incumbent efficiency posture.
Block: Item 1A emphasizes crypto/bitcoin exposure and l